In [9]:
import pandas as pd
import datetime as dt

# 1. Cargar datos y asegurar formato de fecha
df = pd.read_csv('../../../data/thelook/procesado/orders_price_client.csv')

df = df.dropna()
df['created_at'] = pd.to_datetime(df['created_at'])

# 2. Definir fecha de referencia (un día después de la última compra en el dataset)
snapshot_date = df['created_at'].max() + dt.timedelta(days=1)

# 3. Agrupar por usuario para calcular R, F y M
rfm = df.groupby('user_id').agg({
    'created_at': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'sale_price': 'sum'
})

# Renombrar columnas
rfm.rename(columns={
    'created_at': 'Recency',
    'order_id': 'Frequency',
    'sale_price': 'Monetary'
}, inplace=True)

print(rfm.head())

         Recency  Frequency    Monetary
user_id                                
1           1103          2   63.990002
2             21          1   58.990002
3            290          7  458.130003
4             32          5  432.469995
5            245          5  218.490000


In [10]:
from sklearn.preprocessing import StandardScaler

# 1. Inicializar el escalador
scaler = StandardScaler()

# 2. Ajustar y transformar los datos RFM
# Esto convierte los valores en "puntuaciones Z" (cuántas desviaciones estándar se alejan de la media)
rfm_scaled = scaler.fit_transform(rfm)

# 3. Convertirlo de nuevo a un DataFrame para verlo claro
rfm_scaled_df = pd.DataFrame(rfm_scaled, index=rfm.index, columns=rfm.columns)

print("Datos originales (Primeras 5 filas):")
print(rfm.head())
print("\nDatos escalados (Fíjate cómo ahora todos están en un rango similar):")
print(rfm_scaled_df.head())

Datos originales (Primeras 5 filas):
         Recency  Frequency    Monetary
user_id                                
1           1103          2   63.990002
2             21          1   58.990002
3            290          7  458.130003
4             32          5  432.469995
5            245          5  218.490000

Datos escalados (Fíjate cómo ahora todos están en un rango similar):
          Recency  Frequency  Monetary
user_id                               
1        3.272475  -0.164637 -0.513578
2       -0.972778  -0.785981 -0.549868
3        0.082650   2.942084  2.347074
4       -0.929620   1.699396  2.160835
5       -0.093909   1.699396  0.607777


In [11]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

# 1. Definir un rango de búsqueda (por ejemplo, de 2 a 10 clusters)
range_n_clusters = range(2, 11)
best_k = 2
best_score = -1

for k in range_n_clusters:
    # Ejecutar KMeans para cada k
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(rfm_scaled)
    
    # Calcular el Silhouette Score
    silhouette_avg = silhouette_score(rfm_scaled, cluster_labels)
    print(f"Para k={k}, el Silhouette Score es: {silhouette_avg:.4f}")
    
    # Guardar el k que tenga el puntaje más alto
    if silhouette_avg > best_score:
        best_score = silhouette_avg
        best_k = k

print(f"\n✅ El número óptimo de clusters seleccionado automáticamente es: {best_k}")

# 2. Aplicar el modelo final con el 'best_k'
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans_final.fit_predict(rfm_scaled)

Para k=2, el Silhouette Score es: 0.4489
Para k=3, el Silhouette Score es: 0.4246
Para k=4, el Silhouette Score es: 0.4011
Para k=5, el Silhouette Score es: 0.3425
Para k=6, el Silhouette Score es: 0.3365
Para k=7, el Silhouette Score es: 0.3462
Para k=8, el Silhouette Score es: 0.3563
Para k=9, el Silhouette Score es: 0.3349
Para k=10, el Silhouette Score es: 0.3414

✅ El número óptimo de clusters seleccionado automáticamente es: 2


In [12]:
# Ver los promedios de cada métrica por cluster
resumen_clusters = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(2)

print(resumen_clusters)

        Recency Frequency Monetary       
           mean      mean     mean  count
Cluster                                  
0        176.46      4.56   313.54  18282
1        297.28      1.56    79.95  59644
